# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [5]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [6]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [7]:
# Task 1 — Asia: GDP per capita vs life expectancy, 2002 vs 2007
import plotly.graph_objects as go

asia = df.loc[(df['continent'] == 'Asia') & (df['year'].isin([2002, 2007]))].copy()
asia['year'] = asia['year'].astype(str)          # categorical -> exactly 2 colours

fig = px.scatter(
    data_frame=asia, x='gdpPercap', y='lifeExp',
    color='year',
    color_discrete_map={'2002': 'lightgrey', '2007': 'crimson'},   # grey = before, red = after
    category_orders={'year': ['2002', '2007']},
    hover_name='country',
    log_x=True,                                   # wealth is non-linear
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy (years)', 'year': ''},
    template='plotly_white',
    title='Every Asian country lived longer in 2007 than in 2002 - most also grew richer',
)
fig.update_traces(marker=dict(size=9, opacity=0.85))

# Thin grey line per country joins its 2002 point to its 2007 point -> shows the direction of change
before = asia[asia['year'] == '2002'].set_index('country')
after = asia[asia['year'] == '2007'].set_index('country')
for country in before.index:
    fig.add_trace(go.Scatter(
        x=[before.loc[country, 'gdpPercap'], after.loc[country, 'gdpPercap']],
        y=[before.loc[country, 'lifeExp'], after.loc[country, 'lifeExp']],
        mode='lines', line=dict(color='lightgrey', width=1),
        hoverinfo='skip', showlegend=False))
fig.data = fig.data[2:] + fig.data[:2]            # draw lines underneath the dots

fig.update_layout(
    font=dict(family='Arial', size=13), height=550,
    legend=dict(orientation='h', x=0, y=1.08, title=''),
)
fig.show() 

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [8]:
# Task 2 — Bubble chart, 2007: grey everything, highlight Asia
import math

df07 = df.loc[df['year'] == 2007]

highlight = 'Asia'
colour_map = {c: ('crimson' if c == highlight else 'lightgrey') for c in df07['continent'].unique()}

fig = px.scatter(
    data_frame=df07, x='gdpPercap', y='lifeExp',
    size='pop', size_max=60,                      # cap bubble size so small countries stay visible
    color='continent', color_discrete_map=colour_map,
    hover_name='country', hover_data={'continent': False},
    log_x=True,                                   # wealth is non-linear
    opacity=0.7,
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy (years)', 'pop': 'Population'},
    template='plotly_white',
    title='Asia holds the two most populous countries, yet both still sit mid-table on wealth and health',
)

# Annotate the story: the two giants
for country, ax, ay in [('China', 70, -40), ('India', 70, 40)]:
    row = df07[df07['country'] == country].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']), y=row['lifeExp'],
        text=f'{country}<br>{row["pop"] / 1e9:.2f}bn people',
        showarrow=True, arrowhead=1, ax=ax, ay=ay, arrowcolor='crimson',
        font=dict(size=12, family='Arial', color='crimson'),
    )

fig.update_layout(font=dict(family='Arial', size=13), showlegend=False, height=600)
fig.show()